In [1]:
!pip install ramantune


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from ramantune.pipeline.raman_pipeline import RamanPipeline

from ramantune.search.search_space import DenoiserSpace, BaselineSpace, NormalizerSpace, ClassifierSpace, FeatureSelectionSpace
from ramantune.utils.config import DENOISING_STR, BASELINE_STR, NORMALIZE_STR, FEATURE_SELECTION_STR, CLASSIFIER_STR
from ramantune.search.strategies import GridSearchStrategy
from ramantune.search import RamanSearch

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold

In [4]:
df = pd.read_csv("bin/ovarian_small.csv")
groups = df['patient']

y = df['label'].values
X = df.drop(columns=['label', 'patient'])

In [5]:
def setup_param_grid():
  denoiser_list = [
        DenoiserSpace("savgol", {"window_length": [7], "polyorder": [3]}),
        DenoiserSpace(None) # No Denoising
  ]

  baseline_list = [
      BaselineSpace("imodpoly", {"poly_order": [4]}),
      BaselineSpace("asls", {"lam": [100]}),
  ]

  normalization_list = [
      NormalizerSpace("vector"),
  ]

  feature_selection_list = [
      FeatureSelectionSpace(None), # No feature selection
      FeatureSelectionSpace(PCA(),{"n_components": [0.90, 10]}),
  ]

  classifier_list = [
      ClassifierSpace(SVC(),{"C": [0.1, 10], "kernel": ["rbf", "linear"], "gamma": ["scale"]})
  ]

  param_list = {
      DENOISING_STR: denoiser_list,
      BASELINE_STR: baseline_list,
      NORMALIZE_STR: normalization_list,
      FEATURE_SELECTION_STR: feature_selection_list,
      CLASSIFIER_STR: classifier_list
  }

  return param_list

In [6]:
estimator = RamanPipeline()
param_grid = setup_param_grid()

In [7]:
search = RamanSearch(estimator=estimator,
                         research_strategy=GridSearchStrategy(),
                         param_grid=param_grid,
                         cv=StratifiedGroupKFold(n_splits=2, random_state=42, shuffle=True),
                         return_train_score=True,
                         n_jobs=1,
                         verbose=10,
                         refit="accuracy")
res = search.fit(X, y, groups=groups)

Fitting 2 folds for each of 48 candidates, totalling 96 fits
[CV 1/2; 1/48] START baseline__algorithm=imodpoly, baseline__poly_order=4, classifier__C=0.1, classifier__algorithm=SVC(), classifier__kernel=rbf, denoising__algorithm=savgol, denoising__polyorder=3, denoising__window_length=7, feature__algorithm=None, normalize__algorithm=vector
[CV 1/2; 1/48] END baseline__algorithm=imodpoly, baseline__poly_order=4, classifier__C=0.1, classifier__algorithm=SVC(), classifier__kernel=rbf, denoising__algorithm=savgol, denoising__polyorder=3, denoising__window_length=7, feature__algorithm=None, normalize__algorithm=vector; accuracy: (train=0.667, test=0.333) f1: (train=0.400, test=0.250) patient_accuracy: (train=0.667, test=0.333) precision: (train=0.333, test=0.167) recall: (train=0.500, test=0.500) sensitivity: (train=0.500, test=0.500) specificity: (train=0.500, test=0.500) total time=   0.2s
[CV 2/2; 1/48] START baseline__algorithm=imodpoly, baseline__poly_order=4, classifier__C=0.1, classi

In [8]:
print(search.get_best_params())
print(search.get_best_score())

{'baseline__algorithm': 'asls', 'baseline__lam': 100, 'classifier__C': 10, 'classifier__algorithm': SVC(), 'classifier__kernel': 'rbf', 'denoising__algorithm': 'savgol', 'denoising__polyorder': 3, 'denoising__window_length': 7, 'feature__algorithm': PCA(), 'feature__n_components': 10, 'normalize__algorithm': 'vector'}
0.5714285714285714


In [9]:
result_cv = search.get_cv_results(file_path=f"result.csv",
                          return_split_scores=False,
                          return_combined_params=True,
                          round_values=True)

In [10]:
result_cv

,denoising,baseline,normalize,feature,classifier,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_accuracy,...,std_train_sensitivity,split0_test_patient_accuracy,split1_test_patient_accuracy,mean_test_patient_accuracy,std_test_patient_accuracy,rank_test_patient_accuracy,split0_train_patient_accuracy,split1_train_patient_accuracy,mean_train_patient_accuracy,std_train_patient_accuracy
0,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,None,"SVC(C=0.1,kernel=rbf)",0.1923,0.1167,0.1589,0.0219,0.3333,...,0.0000,0.3333,0.3333,0.3333,0.0000,14,0.6667,0.6667,0.6667,0.0000
1,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,None,"SVC(C=0.1,kernel=linear)",0.0880,0.0294,0.1441,0.0369,0.3333,...,0.0000,0.3333,0.3333,0.3333,0.0000,14,0.6667,0.6667,0.6667,0.0000
2,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,None,"SVC(C=10.0,kernel=rbf)",0.1201,0.0086,0.1786,0.0081,0.3810,...,0.0179,0.3333,0.3333,0.3333,0.0000,14,1.0000,1.0000,1.0000,0.0000
3,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,None,"SVC(C=10.0,kernel=linear)",0.1658,0.0521,0.1539,0.0041,0.3333,...,0.1607,0.3333,0.3333,0.3333,0.0000,14,0.6667,1.0000,0.8333,0.1667
4,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,PCA(n_components=0.9),"SVC(C=0.1,kernel=rbf)",0.1279,0.0409,0.1739,0.0013,0.3333,...,0.0000,0.3333,0.3333,0.3333,0.0000,14,0.6667,0.6667,0.6667,0.0000
5,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,PCA(n_components=10.0),"SVC(C=0.1,kernel=rbf)",0.1172,0.0246,0.2097,0.0271,0.3333,...,0.0000,0.3333,0.3333,0.3333,0.0000,14,0.6667,0.6667,0.6667,0.0000
6,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,PCA(n_components=0.9),"SVC(C=0.1,kernel=linear)",0.1327,0.0536,0.1655,0.0524,0.3333,...,0.0000,0.3333,0.3333,0.3333,0.0000,14,0.6667,0.6667,0.6667,0.0000
7,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,PCA(n_components=10.0),"SVC(C=0.1,kernel=linear)",0.2187,0.1158,0.3002,0.0083,0.3333,...,0.0000,0.3333,0.3333,0.3333,0.0000,14,0.6667,0.6667,0.6667,0.0000
8,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,PCA(n_components=0.9),"SVC(C=10.0,kernel=rbf)",0.0814,0.0234,0.1799,0.0039,0.3333,...,0.0000,0.3333,0.6667,0.5000,0.1667,1,1.0000,1.0000,1.0000,0.0000
9,"savgol(polyorder=3.0,window_length=7.0)",imodpoly(poly_order=4.0),vector,PCA(n_components=10.0),"SVC(C=10.0,kernel=rbf)",0.0809,0.0071,0.0898,0.0003,0.3333,...,0.0000,0.3333,0.6667,0.5000,0.1667,1,1.0000,1.0000,1.0000,0.0000
